In [3]:
mid_rate = 17.85       # the fair rate (pesos per dollar)
offered_rate = 17.20   # the rate the provider gave you

spread = (mid_rate - offered_rate) / mid_rate
print(spread)


0.036414565826330646


In [4]:
amount_usd = 300
spread_usd = round(amount_usd * spread, 2)
print(spread_usd)

10.92


In [5]:
funding_method = "cash"
cash_like = ["cash", "money_order", "cashiers_check"]

if funding_method in cash_like:
    tax = round(0.01 * amount_usd, 2)
else:
    tax = 0.0
print(tax)

3.0


In [6]:
fee_usd = 4.99
total = round(fee_usd + spread_usd + tax, 2)
print(total)
print(total / amount_usd)

18.91
0.06303333333333333


In [7]:
lost_local = round(amount_usd * (mid_rate - offered_rate), 2)
print(lost_local)

195.0


In [9]:
print(round(total * 24, 2))


453.84


In [10]:
funding_method = "debit_card"

if funding_method in cash_like:
    tax = round(0.01 * amount_usd, 2)
else:
    tax = 0.0

total_debit = round(fee_usd + spread_usd + tax, 2)
print(total_debit)
print(round((total - total_debit) * 24, 2))

15.91
72.0


In [11]:
def spread_pct(mid_rate, offered_rate):
    return (mid_rate - offered_rate) / mid_rate

print(spread_pct(17.85, 17.20))
print(spread_pct(18.00, 17.50))

0.036414565826330646
0.027777777777777776


In [12]:
def excise_tax(amount_usd, funding_method):
    cash_like = ["cash", "money_order", "cashiers_check"]
    if funding_method in cash_like:
        return round(0.01 * amount_usd, 2)
    return 0.0

print(excise_tax(300, "cash"))
print(excise_tax(300, "debit_card"))
print(excise_tax(500, "money_order"))

3.0
0.0
5.0


In [13]:
def true_cost(amount_usd, fee_usd, offered_rate, mid_rate, funding_method):
    pct = spread_pct(mid_rate, offered_rate)
    spread_usd = round(amount_usd * pct, 2)
    tax = excise_tax(amount_usd, funding_method)
    total = round(fee_usd + spread_usd + tax, 2)
    return {
        "spread_usd": spread_usd,
        "tax": tax,
        "total_cost_usd": total,
        "total_cost_pct": round(total / amount_usd, 4),
    }

print(true_cost(300, 4.99, 17.20, 17.85, "cash"))
print(true_cost(300, 4.99, 17.20, 17.85, "debit_card"))
print(true_cost(200, 0, 17.00, 17.85, "bank_account"))

{'spread_usd': 10.92, 'tax': 3.0, 'total_cost_usd': 18.91, 'total_cost_pct': 0.063}
{'spread_usd': 10.92, 'tax': 0.0, 'total_cost_usd': 15.91, 'total_cost_pct': 0.053}
{'spread_usd': 9.52, 'tax': 0.0, 'total_cost_usd': 9.52, 'total_cost_pct': 0.0476}


In [14]:
print(true_cost(500, 3.99, 86.20, 87.00, "bank_account"))   # India
print(true_cost(200, 3.49, 56.10, 57.00, "debit_card"))     # Philippines
print(true_cost(150, 0, 17.10, 17.60, "cash"))              # Mexico

{'spread_usd': 4.6, 'tax': 0.0, 'total_cost_usd': 8.59, 'total_cost_pct': 0.0172}
{'spread_usd': 3.16, 'tax': 0.0, 'total_cost_usd': 6.65, 'total_cost_pct': 0.0333}
{'spread_usd': 4.26, 'tax': 1.5, 'total_cost_usd': 5.76, 'total_cost_pct': 0.0384}


In [15]:
for bad in [(300, None, 17.20, 17.85, "cash"), (0, 4.99, 17.20, 17.85, "cash")]:
    try:
        true_cost(*bad)
    except Exception as e:
        print(type(e).__name__, "-", e)

print(true_cost(300, 4.99, 17.20, 17.85, "bitcoin"))

TypeError - unsupported operand type(s) for +: 'NoneType' and 'float'
ZeroDivisionError - float division by zero
{'spread_usd': 10.92, 'tax': 0.0, 'total_cost_usd': 15.91, 'total_cost_pct': 0.053}


In [16]:
def true_cost(amount_usd, fee_usd, offered_rate, mid_rate, funding_method):
    if fee_usd is None:
        raise ValueError("fee_usd is missing.")
    if amount_usd <= 0:
        raise ValueError("amount_usd must be greater than 0.")
    allowed = ["cash", "money_order", "cashiers_check",
               "debit_card", "credit_card", "bank_account"]
    if funding_method not in allowed:
        raise ValueError(f"Unknown funding method: {funding_method!r}")

    pct = spread_pct(mid_rate, offered_rate)
    spread_usd = round(amount_usd * pct, 2)
    tax = excise_tax(amount_usd, funding_method)
    total = round(fee_usd + spread_usd + tax, 2)
    return {
        "spread_usd": spread_usd,
        "tax": tax,
        "total_cost_usd": total,
        "total_cost_pct": round(total / amount_usd, 4),
    }

for bad in [(300, None, 17.20, 17.85, "cash"),
            (0, 4.99, 17.20, 17.85, "cash"),
            (300, 4.99, 17.20, 17.85, "bitcoin")]:
    try:
        true_cost(*bad)
    except ValueError as e:
        print("Error:", e)

print(true_cost(300, 4.99, 17.20, 17.85, "cash"))

Error: fee_usd is missing.
Error: amount_usd must be greater than 0.
Error: Unknown funding method: 'bitcoin'
{'spread_usd': 10.92, 'tax': 3.0, 'total_cost_usd': 18.91, 'total_cost_pct': 0.063}


In [17]:
import os, shutil

os.makedirs("remit-check/remitcheck", exist_ok=True)
os.makedirs("remit-check/tests", exist_ok=True)
open("remit-check/remitcheck/__init__.py", "w").close()

shutil.copy("cost.py", "remit-check/remitcheck/cost.py")
shutil.copy("test_cost.py", "remit-check/tests/test_cost.py")

print(os.listdir("remit-check"))

['remitcheck', 'tests']


In [18]:
import sys
!{sys.executable} -m pip install pytest
%cd remit-check
!{sys.executable} -m pytest

C:\Users\suvek\Downloads\remit-check
============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0
rootdir: C:\Users\suvek\Downloads\remit-check
plugins: anyio-4.10.0
collected 19 items

tests\test_cost.py ...................                                   [100%]

============================= 19 passed in 0.06s ==============================


In [2]:
import shutil, os
shutil.copy("rates.py", "remit-check/remitcheck/rates.py")
shutil.copy("test_rates.py", "remit-check/tests/test_rates.py")
print(os.listdir("remit-check/remitcheck"), os.listdir("remit-check/tests"))

['cost.py', 'rates.py', '__init__.py', '__pycache__'] ['test_cost.py', 'test_rates.py', '__pycache__']


In [3]:
import sys
!{sys.executable} -m pip install pandas requests
%cd remit-check
!{sys.executable} -m pytest

C:\Users\suvek\Downloads\remit-check
============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0
rootdir: C:\Users\suvek\Downloads\remit-check
plugins: anyio-4.10.0
collected 26 items

tests\test_cost.py ...................                                   [ 73%]
tests\test_rates.py .......                                              [100%]

============================== warnings summary ===============================
tests/test_rates.py::test_cache_miss_fetches_and_returns_rate
tests/test_rates.py::test_second_call_uses_cache_not_api
tests/test_rates.py::test_one_fetch_caches_all_three_currencies
tests/test_rates.py::test_weekend_date_uses_prior_business_day_rate
  C:\Users\suvek\Downloads\remit-check\remitcheck\rates.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when det

In [4]:
from remitcheck.rates import get_mid_rate

print(get_mid_rate("MXN", "2026-03-13"))   # a Friday
print(get_mid_rate("MXN", "2026-03-14"))   # a Saturday

C:\Users\suvek\Downloads\remit-check\remitcheck\rates.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)


17.8213
17.8213
